#  Deep Neural Network for m-Height Estimation

Build a Deep Neural Network that predicts the **m-height** of an Analog Code given:
- Three integers `(n, k, m)`
- A `k × (n-k)` parity matrix `P` (the non-identity part of `G = [I_k | P]`)

## My Approach: Deep Residual Network with Cumsum Output

The key insight is that **m-heights are monotonically increasing**: for a fixed `(n, k, P)`,
$$1 = h_0 \le h_1 \le h_2 \le \ldots \le h_{n-k}$$

Instead of training the network to predict a single m-height for a given `(n, k, m, P)`, I let the network predict **all 5 m-heights at once** for a given `(n, k, P)`, using a cumsum output that structurally guarantees monotonicity.

### Architecture Flow
```
Input (n, k, P) → LayerNorm → Dense(1024)
                → 4× ResBlock(1024) with LayerNorm + GELU + Dropout(0.15)
                → Dense(512, gelu)
                → Dense(5, softplus)
                → Cumsum  →  log2(m-heights) for m=1..5
```


In [ ]:
from google.colab import drive

print("Connecting to Google Drive...")
drive.mount('/content/drive')

Connecting to Google Drive...
Mounted at /content/drive


## STEP 1: Import Libraries and Setup
Initializes the Python environment by importing required deep learning libraries (PyTorch, NumPy, Pickle). Random seeds are explicitly set across all libraries to ensure strict computational reproducibility. Hardware acceleration is detected and configured to route tensor operations to the GPU.

In [ ]:
import numpy as np
import pickle
import os
from collections import defaultdict
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

SEED = 2026
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"PyTorch version: {torch.__version__}")
print(f"Using device: {DEVICE}")
if DEVICE.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")

PyTorch version: 2.10.0+cu128
Using device: cuda
GPU: NVIDIA A100-SXM4-40GB


## STEP 2: Build the DNN Architecture
Defines the core Deep Neural Network.
* **Pre-Activation Residual Blocks:** To prevent vanishing gradients across deep layers, the network utilizes a `LayerNorm -> Dense -> GELU -> Dropout -> Dense` skip-connection design.
* **Structural Inductive Bias:** The output head transitions from a standard linear layer to a 5-way Multi-Task output. By applying a `Softplus` activation to generate non-negative increments, followed by a `Cumsum` operation, the network structurally enforces the mathematical rule that $m$-heights must monotonically increase.

In [ ]:
class ResBlock(nn.Module):
    """Pre-activation residual block: LayerNorm -> Linear -> GELU -> Dropout -> Linear -> +residual"""
    def __init__(self, dim=1024, dropout=0.15):
        super(ResBlock, self).__init__()
        self.norm = nn.LayerNorm(dim)
        self.fc1 = nn.Linear(dim, dim)
        self.fc2 = nn.Linear(dim, dim)
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        residual = x
        out = self.norm(x)
        out = F.gelu(self.fc1(out))
        out = self.drop(out)
        out = self.fc2(out)
        return out + residual


In [ ]:
class AnalogCodePredictor(nn.Module):
    """Deep Residual Network with cumsum output for monotonic m-height prediction.

    Monotonicity (h_1 <= h_2 <= ... <= h_5) is structurally guaranteed by
    softplus + cumsum.
    """
    def __init__(self, input_dim=32, hidden_dim=1024, num_blocks=4, dropout=0.15, num_m=5):
        super(AnalogCodePredictor, self).__init__()
        self.num_m = num_m

        # Entry: normalize input -> project to hidden
        self.input_norm = nn.LayerNorm(input_dim)
        self.entry_dense = nn.Linear(input_dim, hidden_dim)

        # 4 stacked residual blocks
        self.resblocks = nn.Sequential(*[
            ResBlock(dim=hidden_dim, dropout=dropout)
            for _ in range(num_blocks)
        ])

        # Exit head: hidden -> 512 -> num_m
        self.exit_dense = nn.Linear(hidden_dim, 512)
        self.gap_head = nn.Linear(512, num_m)

    def forward(self, x):
        x = self.input_norm(x)
        x = F.gelu(self.entry_dense(x))
        x = self.resblocks(x)
        x = F.gelu(self.exit_dense(x))

        # Predict 5 non-negative gaps in log2-space
        gaps = F.softplus(self.gap_head(x))   # shape (batch, 5), all >= 0

        # Cumsum gives monotonic log2(m-heights)
        log2_heights = torch.cumsum(gaps, dim=-1)   # shape (batch, 5)

        return log2_heights

## STEP 3: Loss Function
Establishes a custom objective function tailored to the multi-task logarithmic grading metric. The `MaskedLog2MSELoss` computes the Mean Squared Error in logarithmic space. A masking mechanism is incorporated to ensure the network is only penalized for $m$-heights that explicitly exist for a given matrix configuration, gracefully handling missing labels.

In [ ]:
class MaskedLog2MSELoss(nn.Module):
    """MSE loss in log2-space with masking for missing m values."""
    def __init__(self):
        super(MaskedLog2MSELoss, self).__init__()

    def forward(self, log2_pred, log2_true, mask):
        """
        Args:
            log2_pred: (batch, 5) — predicted log2 m-heights
            log2_true: (batch, 5) — true log2 m-heights (any value where mask=0)
            mask:      (batch, 5) — 1.0 if this m is valid, 0.0 if missing
        """
        squared_error = (log2_pred - log2_true) ** 2
        masked_error = squared_error * mask

        # Average over valid entries only
        n_valid = mask.sum().clamp(min=1.0)
        return masked_error.sum() / n_valid

## STEP 4: Load and Group Training Data

The original data file has each sample as `(n, k, m, P) → y`. Since my new architecture predicts all m values for a given `(n, k, P)`, I need to **group samples that share the same P** and assemble all available m-heights together.

### Grouping strategy
1. For each sample `(n, k, m, P) → y`, hash the P matrix to a key
2. Build a dictionary: `(n, k, P_key) -> {m: y_for_this_m, ...}`
3. After grouping, each unique `(n, k, P)` combo has up to `n-k` m-heights collected

### Output format per sample
Each grouped sample becomes:
- **Input X** (32-dim): `[n, k, P_padded(30)]` — note: NO m here
- **Target y** (5-dim): `[log2(h_1), log2(h_2), log2(h_3), log2(h_4), log2(h_5)]`
- **Mask** (5-dim): 1.0 where the m value is available, 0.0 where missing

Note: index `i` in the output corresponds to `m = i + 1`. So index 1 is m=2, index 2 is m=3, etc.

### NOTE:  if some m values are missing?
The masked loss handles it. If only m=2 and m=3 are available for a sample, only those positions contribute to the loss. The model still learns to predict all 5 from these partial labels because of multi-task sharing across thousands of samples.

In [ ]:
# File paths
FEATURES_FILE = "/content/drive/MyDrive/MoE_Project/MERGED_features.pkl"
LABELS_FILE = "/content/drive/MyDrive/MoE_Project/MERGED_labels.pkl"

print("Loading raw datasets...")
with open(FEATURES_FILE, 'rb') as f:
    features_raw = pickle.load(f)
with open(LABELS_FILE, 'rb') as f:
    labels_raw = pickle.load(f)

print(f"Loaded {len(features_raw):,} raw samples")
print("Grouping by (n, k, P) — collecting all m-heights per matrix...")

grouped = defaultdict(dict)

for record, y in zip(features_raw, labels_raw):
    n = int(record[0])
    k = int(record[1])
    m = int(record[2])
    P = np.array(record[3], dtype=np.float32)

    P_key = (n, k, P.tobytes())
    grouped[P_key][m] = float(y)

print(f"Number of unique (n, k, P) groups: {len(grouped):,}")

NUM_M = 5  # m can be 1, 2, 3, 4, 5
X_list = []
y_list = []
mask_list = []

for (n, k, P_bytes), m_to_y in grouped.items():
    # Reconstruct P from bytes
    P_flat = np.frombuffer(P_bytes, dtype=np.float32).copy()

    # Build input vector
    P_padded = np.zeros(30, dtype=np.float32)
    P_padded[:len(P_flat)] = P_flat
    x_vec = np.concatenate(([n, k], P_padded)).astype(np.float32)

    # Build target and mask (5-dim, indexed by m=1..5 -> idx 0..4)
    y_vec = np.zeros(NUM_M, dtype=np.float32)
    mask_vec = np.zeros(NUM_M, dtype=np.float32)

    for m, y_val in m_to_y.items():
        idx = m - 1  # m=1 -> idx 0, m=2 -> idx 1, etc.
        if 0 <= idx < NUM_M:
            y_vec[idx] = np.log2(max(y_val, 1.0))
            mask_vec[idx] = 1.0

    X_list.append(x_vec)
    y_list.append(y_vec)
    mask_list.append(mask_vec)

X_tensor = torch.tensor(np.array(X_list), dtype=torch.float32)
y_tensor = torch.tensor(np.array(y_list), dtype=torch.float32)
mask_tensor = torch.tensor(np.array(mask_list), dtype=torch.float32)

print(f"X tensor:    {X_tensor.shape}")
print(f"y tensor:    {y_tensor.shape}  (log2 m-heights)")
print(f"mask tensor: {mask_tensor.shape}")
print(f"Average labels per sample: {mask_tensor.sum(dim=1).mean().item():.2f} (out of 5)")

Loading raw datasets...
Loaded 2,020,563 raw samples
Grouping by (n, k, P) — collecting all m-heights per matrix...
Number of unique (n, k, P) groups: 701,462
X tensor:    torch.Size([701462, 32])
y tensor:    torch.Size([701462, 5])  (log2 m-heights)
mask tensor: torch.Size([701462, 5])
Average labels per sample: 2.85 (out of 5)


## STEP 5: Hyperparameters and Setup

### Final Hyperparameters

| Hyperparameter | Value | Why |
|----------------|-------|-----|
| Hidden dim | 1024 | Reference architecture default |
| Residual blocks | 4 | Reference architecture default |
| Dropout | 0.15 | Multi-task learning provides natural regularization |
| Output dim | 5 | One per m value (1-5) |
| Batch size | 1024 | Stable gradients without slowing down too much |
| Learning rate | 1e-3 | Standard for AdamW + warmup |
| Weight decay | 0.005 | Light regularization |
| Epochs | 200 | Multi-task converges well by then |
| LR schedule | CosineAnnealing | Smooth decay improves final convergence |
| Loss | Masked MSE in log2 | Matches output format, handles missing m's |
| Optimizer | AdamW | Decoupled weight decay |
| AdamW betas | (0.9, 0.95) | Faster adaptation |

In [ ]:
# Hyperparameters
EPOCHS = 200
BATCH_SIZE = 1024
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 0.005
GRAD_CLIP = 1.0
SAVE_PATH = "/content/drive/MyDrive/MoE_Project/mheight_model.pt"

# DataLoader
dataset = TensorDataset(X_tensor, y_tensor, mask_tensor)
loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True
)

# Model
model = AnalogCodePredictor(
    input_dim=32,
    hidden_dim=1024,
    num_blocks=4,
    dropout=0.15,
    num_m=5
).to(DEVICE)

# Optimizer
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    betas=(0.9, 0.95)
)

# Cosine annealing scheduler
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=EPOCHS,
    eta_min=1e-6
)

criterion = MaskedLog2MSELoss()

n_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {n_params:,}")
print(f"Steps per epoch:  {len(loader)}")
print(f"Total training steps: {EPOCHS * len(loader):,}")

Model parameters: 8,966,213
Steps per epoch:  686
Total training steps: 137,200


## STEP 6: Training

In this step, we train the model using an iterative optimization process to handle the complex, non-convex nature of the m-height function.

- **AdamW Optimizer:**  
  We use AdamW, which separates weight decay from gradient updates. This helps improve generalization and makes training more stable.

- **Gradient Clipping:**  
  To prevent unstable or extremely large gradients (especially in difficult cases like (9,5,4)), we apply gradient clipping. This keeps updates controlled and avoids divergence.

- **Cosine Annealing Scheduler:**  
  The learning rate is adjusted using cosine annealing. It starts higher for better exploration and gradually decreases to allow smooth convergence.

- **Checkpointing:**  
  The model is saved only when it achieves better overall performance than before, ensuring we keep the best version during training.

In [ ]:
print(f"Starting training: {EPOCHS} epochs, batch={BATCH_SIZE}, lr={LEARNING_RATE}")
print(f"Multi-task: predicting 5 m-heights simultaneously")
print("-" * 70)

best_loss = float('inf')
model.train()

for epoch in range(EPOCHS):
    total_loss = 0.0
    n_samples = 0

    for batch_X, batch_y, batch_mask in loader:
        batch_X = batch_X.to(DEVICE, non_blocking=True)
        batch_y = batch_y.to(DEVICE, non_blocking=True)
        batch_mask = batch_mask.to(DEVICE, non_blocking=True)

        # Forward pass
        optimizer.zero_grad()
        log2_pred = model(batch_X)
        loss = criterion(log2_pred, batch_y, batch_mask)

        # Backward pass
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=GRAD_CLIP)
        optimizer.step()

        total_loss += loss.item() * batch_X.size(0)
        n_samples += batch_X.size(0)

    # Step scheduler at end of epoch
    scheduler.step()
    avg_loss = total_loss / n_samples
    if avg_loss < best_loss:
        best_loss = avg_loss
        torch.save(model.state_dict(), SAVE_PATH)

    if (epoch + 1) % 10 == 0 or epoch == 0:
        current_lr = scheduler.get_last_lr()[0]
        print(f"   Epoch [{epoch+1:3d}/{EPOCHS}] | Loss: {avg_loss:.5f} | "
              f"Best: {best_loss:.5f} | LR: {current_lr:.6f}")

print("-" * 70)
print(f"Training complete. Best loss: {best_loss:.5f}")
print(f"Best model saved to: {SAVE_PATH}")

Starting training: 200 epochs, batch=1024, lr=0.001
Multi-task: predicting 5 m-heights simultaneously
----------------------------------------------------------------------
   Epoch [  1/200] | Loss: 6.96007 | Best: 6.96007 | LR: 0.001000
   Epoch [ 10/200] | Loss: 0.82335 | Best: 0.82335 | LR: 0.000994
   Epoch [ 20/200] | Loss: 0.71998 | Best: 0.71998 | LR: 0.000976
   Epoch [ 30/200] | Loss: 0.68325 | Best: 0.68325 | LR: 0.000946
   Epoch [ 40/200] | Loss: 0.66962 | Best: 0.66962 | LR: 0.000905
   Epoch [ 50/200] | Loss: 0.66150 | Best: 0.65895 | LR: 0.000854
   Epoch [ 60/200] | Loss: 0.65120 | Best: 0.65120 | LR: 0.000794
   Epoch [ 70/200] | Loss: 0.64479 | Best: 0.64479 | LR: 0.000727
   Epoch [ 80/200] | Loss: 0.63823 | Best: 0.63823 | LR: 0.000655
   Epoch [ 90/200] | Loss: 0.63855 | Best: 0.63304 | LR: 0.000579
   Epoch [100/200] | Loss: 0.62754 | Best: 0.62754 | LR: 0.000501
   Epoch [110/200] | Loss: 0.63260 | Best: 0.62130 | LR: 0.000422
   Epoch [120/200] | Loss: 0.61481 

## STEP 7: Final Inference Cell

The model is designed to take a fixed-size input vector and produce multiple outputs corresponding to different values of \( m \).

### Architecture Overview

- **Input Layer (32-dim):**  
  Each sample is represented as a 32-dimensional vector formed using \( n \), \( k \), and the padded sequence \( P \). This ensures all inputs have a consistent size.

- **Hidden Layers:**  
  The input is passed through multiple fully connected (linear) layers. Each layer applies a transformation followed by a non-linear activation function (such as ReLU). These layers help the model learn complex patterns and relationships in the data.

- **Output Layer (5-dim):**  
  The final layer produces a vector of size 5:
  \[
  [\log_2(h_1), \log_2(h_2), \log_2(h_3), \log_2(h_4), \log_2(h_5)]
  \]
  Each value corresponds to a different \( m \) (from 1 to 5).

### Key Idea

Instead of training separate models for each \( m \), this model predicts all 5 values at once. During inference, we simply select the required output based on the given \( m \).

In [ ]:

import numpy as np
import pickle
import os
import torch
import torch.nn as nn
import torch.nn.functional as F

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Inference device: {DEVICE}")

class ResBlock(nn.Module):
    def __init__(self, dim=1024, dropout=0.15):
        super().__init__()
        self.norm = nn.LayerNorm(dim)
        self.fc1 = nn.Linear(dim, dim)
        self.fc2 = nn.Linear(dim, dim)
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        residual = x
        out = self.norm(x)
        out = F.gelu(self.fc1(out))
        out = self.drop(out)
        out = self.fc2(out)
        return out + residual

class AnalogCodePredictor(nn.Module):
    def __init__(self, input_dim=32, hidden_dim=1024, num_blocks=4, dropout=0.15, num_m=5):
        super().__init__()
        self.num_m = num_m
        self.input_norm = nn.LayerNorm(input_dim)
        self.entry_dense = nn.Linear(input_dim, hidden_dim)
        self.resblocks = nn.Sequential(*[
            ResBlock(dim=hidden_dim, dropout=dropout)
            for _ in range(num_blocks)
        ])
        self.exit_dense = nn.Linear(hidden_dim, 512)
        self.gap_head = nn.Linear(512, num_m)

    def forward(self, x):
        x = self.input_norm(x)
        x = F.gelu(self.entry_dense(x))
        x = self.resblocks(x)
        x = F.gelu(self.exit_dense(x))
        gaps = F.softplus(self.gap_head(x))
        log2_heights = torch.cumsum(gaps, dim=-1)
        return log2_heights

class MaskedLog2MSELoss(nn.Module):
    """MSE loss in log2-space with masking for missing m values."""
    def __init__(self):
        super(MaskedLog2MSELoss, self).__init__()

    def forward(self, log2_pred, log2_true, mask):
        """
        Args:
            log2_pred: (batch, 5) — predicted log2 m-heights
            log2_true: (batch, 5) — true log2 m-heights (any value where mask=0)
            mask:      (batch, 5) — 1.0 if this m is valid, 0.0 if missing
        """
        squared_error = (log2_pred - log2_true) ** 2
        masked_error = squared_error * mask

        n_valid = mask.sum().clamp(min=1.0)
        return masked_error.sum() / n_valid

def predict_m_heights(test_file, output_file, model_path):
    """Load test data, run model, save predictions to pickle file."""

    # 1. Load model
    if not os.path.exists(model_path):
        raise FileNotFoundError(f"Model weights not found at {model_path}")

    model = AnalogCodePredictor().to(DEVICE)
    model.load_state_dict(torch.load(model_path, map_location=DEVICE, weights_only=True))
    model.eval()
    print(f"Model loaded: {model_path}")

    # 2. Load test data
    with open(test_file, 'rb') as f:
        test_raw = pickle.load(f)
    N_TEST = len(test_raw)
    print(f"Loaded {N_TEST:,} test samples from: {test_file}")

    # 3. Build 32-dim inputs and remember which m to pick for each sample
    print("Preprocessing test samples...")
    X_test = np.zeros((N_TEST, 32), dtype=np.float32)
    m_indices = np.zeros(N_TEST, dtype=np.int64)

    for i, record in enumerate(test_raw):
        n = int(record[0])
        k = int(record[1])
        m = int(record[2])
        P = record[3]

        P_flat = np.array(P, dtype=np.float32).flatten()
        P_padded = np.zeros(30, dtype=np.float32)
        P_padded[:len(P_flat)] = P_flat

        X_test[i] = np.concatenate(([n, k], P_padded))
        m_indices[i] = m - 1   # m=1 -> idx 0, m=2 -> idx 1, etc.

    print("Running batched inference (batch size 8192)...")
    BATCH = 8192
    final_predictions = np.zeros(N_TEST, dtype=np.float32)

    with torch.no_grad():
        for start in range(0, N_TEST, BATCH):
            end = min(start + BATCH, N_TEST)
            xb = torch.from_numpy(X_test[start:end]).to(DEVICE, non_blocking=True)
            mb = torch.from_numpy(m_indices[start:end]).to(DEVICE)
            log2_all = model(xb)
            log2_pred = log2_all.gather(1, mb.unsqueeze(1)).squeeze(1)

            # Convert log2 -> m-height
            y_pred = torch.pow(2.0, log2_pred)

            final_predictions[start:end] = y_pred.cpu().numpy()

    final_predictions = np.maximum(final_predictions, 1.0)
    final_list = [float(p) for p in final_predictions]

    with open(output_file, 'wb') as f:
        pickle.dump(final_list, f)

    print(f"\nSUCCESS: Saved {N_TEST:,} predictions to: {output_file}")
    print(f"Stats — min: {final_predictions.min():.4f}, "
          f"max: {final_predictions.max():.4f}, "
          f"mean: {final_predictions.mean():.4f}")
    return final_list

TEST_FILE = '/content/Test-n_k_m_P'
OUTPUT_FILE = '/content/Test-mHeights'
MODEL_PATH = '/content/mheight_model.pt'

predictions = predict_m_heights(TEST_FILE, OUTPUT_FILE, MODEL_PATH)
print("=" * 60)
print("INFERENCE COMPLETE")
print("=" * 60)